# Semantic Segmentation

Image classification assigns a single label to an entire image. **Semantic segmentation** is a denser task: we assign a class label to *every* pixel, producing a prediction map of the same spatial resolution as the input. Suppose you want to build an autonomous driving system or a medical imaging pipeline — knowing which class each pixel belongs to is essential, and no single image-level label can capture that.

We implement a **U-Net** [@unet] with a ResNet-18 encoder for 3-class segmentation on the Oxford-IIIT Pet dataset [@pets]: foreground (pet), background, and uncertain/boundary. The core architectural idea is an **encoder-decoder** with **skip connections** — the encoder progressively compresses spatial resolution while building rich feature representations, and the decoder symmetrically recovers spatial detail by fusing coarse encoder features with fine-grained ones from earlier layers. We also study **Dice loss**, a loss function designed for imbalanced pixel-class distributions that complements the standard cross-entropy.

CNN primitives and data augmentation patterns were covered in [NB06](../06-cnn.html). The ResNet-18 architecture — `BasicBlock` and the `ResNet` class — was built from scratch in [NB12](../12-appendix-resnet.html); we reuse those definitions here as the encoder backbone.

<br>

In [ ]:
import math
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms.functional as TF
from pathlib import Path
from matplotlib_inline import backend_inline
from torch.utils.data import DataLoader, Dataset, random_split
from torch.optim.lr_scheduler import OneCycleLR
from torchvision import transforms

DATASET_DIR = Path("./data").resolve()
DATASET_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 42
DEBUG = False
MATPLOTLIB_FORMAT = "png" if DEBUG else "svg"

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
warnings.simplefilter(action="ignore")
backend_inline.set_matplotlib_formats(MATPLOTLIB_FORMAT)

DEVICE = (
    torch.device("cuda:0") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)
print(f"Device: {DEVICE}")

## Dataset

The Oxford-IIIT Pet dataset [@pets] contains 7,349 images of 37 pet breeds with pixel-level trimap annotations. Each annotation mask takes one of three integer values: 1 = foreground (the pet), 2 = background, 3 = uncertain/boundary. We treat this as a 3-class segmentation problem and remap mask values to $\{0, 1, 2\}$ to match PyTorch's zero-indexed class convention.

Images are resized to $256 \times 256$. A key requirement for segmentation is **joint spatial augmentation**: any geometric transformation applied to an image must be applied identically to its mask, so that pixel-level alignment is preserved. We implement this by applying the same random horizontal flip and random crop to both the image and the mask using `torchvision.transforms.functional` with a shared random state.

Defining the dataset class:

In [ ]:
class PetsDataset(Dataset):
    """Oxford-IIIT Pet dataset with joint spatial augmentation."""

    IMG_MEAN = (0.485, 0.456, 0.406)
    IMG_STD  = (0.229, 0.224, 0.225)

    def __init__(self, split="trainval", img_size=256, augment=False):
        self.img_size = img_size
        self.augment  = augment

        self.base = torchvision.datasets.OxfordIIITPet(  # <1>
            root=DATASET_DIR,
            split=split,
            target_types="segmentation",
            download=True,
        )
        self.normalize = transforms.Normalize(self.IMG_MEAN, self.IMG_STD)

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, mask = self.base[idx]                         # <2>

        # Resize both to img_size x img_size
        img  = TF.resize(img,  [self.img_size, self.img_size])
        mask = TF.resize(mask, [self.img_size, self.img_size],
                         interpolation=TF.InterpolationMode.NEAREST)  # <3>

        # Joint augmentation: identical random transforms on image and mask
        if self.augment:
            if random.random() > 0.5:                      # <4>
                img  = TF.hflip(img)
                mask = TF.hflip(mask)

            i, j, h, w = transforms.RandomCrop.get_params(
                img, output_size=(self.img_size, self.img_size)
            )                                              # <5>
            img  = TF.crop(img,  i, j, h, w)
            mask = TF.crop(mask, i, j, h, w)

        img  = self.normalize(TF.to_tensor(img))           # <6>
        mask = torch.as_tensor(np.array(mask), dtype=torch.long) - 1  # <7>

        return img, mask

1. `torchvision.datasets.OxfordIIITPet` with `target_types="segmentation"` returns `(PIL image, PIL mask)` pairs.
2. `mask` is a PIL image with integer pixel values $\{1, 2, 3\}.$
3. Mask pixels are class indices — we use nearest-neighbor interpolation to avoid introducing spurious fractional class values when resizing.
4. Horizontal flip applied to both image and mask with the same 50% probability.
5. `RandomCrop.get_params` samples the top-left corner `(i, j)` and dimensions `(h, w)` once; the same crop is applied to both `img` and `mask`.
6. Convert PIL image to a `[0, 1]` tensor, then normalize with ImageNet statistics.
7. Shift mask values from $\{1, 2, 3\}$ to $\{0, 1, 2\}$ for zero-indexed cross-entropy.

Building train and validation splits, then wrapping in `DataLoader`s:

In [ ]:
BATCH_SIZE = 16
IMG_SIZE   = 256
NUM_CLASSES = 3

full_train = PetsDataset(split="trainval", img_size=IMG_SIZE, augment=True)
test_ds    = PetsDataset(split="test",     img_size=IMG_SIZE, augment=False)

# 80 / 20 split for training and validation
n_val   = int(0.2 * len(full_train))
n_train = len(full_train) - n_val
train_ds, val_ds = random_split(
    full_train, [n_train, n_val],
    generator=torch.Generator().manual_seed(RANDOM_SEED)
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

Visualizing sample image–mask pairs from the training set:

In [ ]:
#| code-fold: true
#| label: fig-pets-samples
#| fig-cap: "Four image–mask pairs from the Oxford-IIIT Pet dataset. Mask colors: green = foreground (pet), gray = background, orange = uncertain/boundary."

MASK_CMAP = mcolors.ListedColormap(["#888888", "#4caf50", "#ff9800"])  # bg, fg, boundary

fig, axes = plt.subplots(4, 2, figsize=(6, 10))
img_mean = torch.tensor(PetsDataset.IMG_MEAN).view(3, 1, 1)
img_std  = torch.tensor(PetsDataset.IMG_STD).view(3, 1, 1)

sample_indices = [0, 1, 2, 3]
for row, idx in enumerate(sample_indices):
    img_t, mask_t = full_train[idx]
    img_np = (img_t * img_std + img_mean).clamp(0, 1).permute(1, 2, 0).numpy()

    axes[row, 0].imshow(img_np)
    axes[row, 0].axis("off")
    if row == 0:
        axes[row, 0].set_title("Image", fontsize=9)

    axes[row, 1].imshow(mask_t.numpy(), cmap=MASK_CMAP, vmin=0, vmax=2)
    axes[row, 1].axis("off")
    if row == 0:
        axes[row, 1].set_title("Mask", fontsize=9)

fig.tight_layout()
plt.show();

## Evaluation metrics

Per-pixel accuracy counts the fraction of correctly classified pixels. While simple, it is biased toward the dominant class: a model that always predicts background will score high accuracy on a dataset where backgrounds occupy most pixels. A better metric for segmentation is **Intersection over Union (IoU)**.

For class $c$, IoU measures the ratio of the overlap between the predicted region $\hat{P}_c$ and the ground-truth region $G_c$ to their union:

$$
\text{IoU}_c = \frac{|\hat{P}_c \cap G_c|}{|\hat{P}_c \cup G_c|} = \frac{\text{TP}_c}{\text{TP}_c + \text{FP}_c + \text{FN}_c}
$$

where $\text{TP}_c$ counts pixels correctly labeled $c$, $\text{FP}_c$ counts pixels incorrectly labeled $c$, and $\text{FN}_c$ counts pixels of class $c$ that were missed. The denominator $\text{TP}_c + \text{FP}_c + \text{FN}_c = |\hat{P}_c \cup G_c|$ follows from the $2\times 2$ confusion matrix:

$$
|\hat{P}_c| + |G_c| - |\hat{P}_c \cap G_c| = (\text{TP}_c + \text{FP}_c) + (\text{TP}_c + \text{FN}_c) - \text{TP}_c = \text{TP}_c + \text{FP}_c + \text{FN}_c.
$$

**Mean IoU (mIoU)** averages $\text{IoU}_c$ over all $C$ classes, giving equal weight to each class regardless of its pixel frequency. A class that is rarely present but badly predicted will pull mIoU down, making it a strict and informative metric.

Implementing `compute_metrics` for per-pixel accuracy and mIoU:

In [ ]:
@torch.inference_mode()
def compute_metrics(preds: torch.Tensor, targets: torch.Tensor, num_classes: int):
    """Compute per-pixel accuracy and mIoU.

    Args:
        preds:   (B, H, W) integer class predictions.
        targets: (B, H, W) integer class targets.
        num_classes: number of classes C.

    Returns:
        accuracy: float, fraction of correctly classified pixels.
        miou:     float, mean IoU across all classes.
        iou_per_class: list[float] of length C.
    """
    correct = (preds == targets).sum().item()
    total   = targets.numel()
    accuracy = correct / total

    iou_per_class = []
    for c in range(num_classes):
        pred_c   = (preds   == c)                # <1>
        target_c = (targets == c)
        tp = (pred_c & target_c).sum().item()    # <2>
        fp = (pred_c & ~target_c).sum().item()
        fn = (~pred_c & target_c).sum().item()
        denom = tp + fp + fn
        iou_per_class.append(tp / denom if denom > 0 else float("nan"))  # <3>

    valid = [v for v in iou_per_class if not math.isnan(v)]
    miou  = sum(valid) / len(valid) if valid else 0.0
    return accuracy, miou, iou_per_class

1. Binary indicator masks for class $c$: `pred_c[i,j]` is `True` iff the model predicted class $c$ at pixel $(i, j).$
2. $\text{TP}_c$ = predicted $c$ AND ground truth $c$; $\text{FP}_c$ = predicted $c$ AND ground truth $\neq c$; $\text{FN}_c$ = not predicted $c$ AND ground truth $c.$
3. If a class is absent from both predictions and targets in a batch, the IoU is undefined — we skip it rather than recording 0.

## U-Net architecture

U-Net [@unet] has a symmetric encoder-decoder structure. The **encoder** (or contracting path) progressively reduces spatial resolution while increasing feature channels, building a hierarchy of feature maps at multiple scales. The **decoder** (or expansive path) symmetrically upsamples back to the input resolution. At each resolution level, the corresponding encoder feature map is concatenated to the decoder input via a **skip connection** — this allows the decoder to recover fine-grained spatial detail that is lost during downsampling.

We use a pretrained ResNet-18 as the encoder backbone, extracting feature maps at 5 spatial scales. Given a $256 \times 256$ input:

| Stage | Layer | Output shape | Stride from input |
|---|---|---|---|
| `s0` | stem (conv + bn + relu) | $B \times 64 \times 128 \times 128$ | $/2$ |
| `s1` | `layer1` | $B \times 64 \times 64 \times 64$ | $/4$ |
| `s2` | `layer2` | $B \times 128 \times 32 \times 32$ | $/8$ |
| `s3` | `layer3` | $B \times 256 \times 16 \times 16$ | $/16$ |
| `s4` | `layer4` | $B \times 512 \times 8 \times 8$ | $/32$ |

The decoder then applies four `DecoderBlock`s in sequence, each doubling the spatial resolution via a transposed convolution, concatenating the corresponding skip feature, and fusing with two convolutional layers. A final $1 \times 1$ convolution maps the 64-channel feature map to $C = 3$ class logits at full $256 \times 256$ resolution.

Defining the decoder block:

In [ ]:
class DecoderBlock(nn.Module):
    """Upsample + concatenate skip connection + two conv-BN-ReLU layers."""

    def __init__(self, in_ch: int, skip_ch: int, out_ch: int):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)  # <1>
        self.conv = nn.Sequential(
            nn.Conv2d(out_ch + skip_ch, out_ch, 3, padding=1, bias=False),    # <2>
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.up(x)                                                        # <3>
        x = torch.cat([x, skip], dim=1)                                       # <4>
        return self.conv(x)

1. `ConvTranspose2d` with `kernel_size=2, stride=2` doubles spatial dimensions: $(B, \text{in\_ch}, H, W) \to (B, \text{out\_ch}, 2H, 2W).$ See the Appendix for how this operation works.
2. After concatenating with the skip, the number of input channels to the first convolution is `out_ch + skip_ch`.
3. Upsample the lower-resolution decoder feature map before fusion.
4. Concatenate along the channel dimension: skip features carry high-resolution spatial detail from the encoder.

Wrapping ResNet-18 as a multi-scale feature extractor:

In [ ]:
class ResNetEncoder(nn.Module):
    """Extract multi-scale features from a ResNet-18 backbone."""

    def __init__(self, pretrained: bool = True):
        super().__init__()
        weights = torchvision.models.ResNet18_Weights.DEFAULT if pretrained else None
        backbone = torchvision.models.resnet18(weights=weights)  # <1>

        # Decompose backbone into named stages
        self.stem   = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)  # /2
        self.pool   = backbone.maxpool                                             # /4
        self.layer1 = backbone.layer1   # 64 ch,  /4
        self.layer2 = backbone.layer2   # 128 ch, /8
        self.layer3 = backbone.layer3   # 256 ch, /16
        self.layer4 = backbone.layer4   # 512 ch, /32

    def forward(self, x: torch.Tensor) -> list:
        """Return feature maps [s0, s1, s2, s3, s4] at /2, /4, /8, /16, /32."""
        s0 = self.stem(x)    # (B, 64,  H/2,  W/2)  # <2>
        s1 = self.layer1(self.pool(s0))  # (B, 64,  H/4,  W/4)
        s2 = self.layer2(s1) # (B, 128, H/8,  W/8)
        s3 = self.layer3(s2) # (B, 256, H/16, W/16)
        s4 = self.layer4(s3) # (B, 512, H/32, W/32)
        return [s0, s1, s2, s3, s4]

1. Using `torchvision.models.ResNet18_Weights.DEFAULT` loads ImageNet-pretrained weights. The pretrained encoder provides much richer initial features than random initialization — especially important when the target dataset is small.
2. We expose `s0` (post-stem, pre-pool) as a high-resolution skip feature at $/2$ spatial stride. This gives the decoder access to fine spatial detail for the final upsampling step.

Assembling the full U-Net:

In [ ]:
class UNet(nn.Module):
    """U-Net with ResNet-18 encoder for semantic segmentation."""

    def __init__(self, num_classes: int = 3, pretrained: bool = True):
        super().__init__()
        self.encoder  = ResNetEncoder(pretrained=pretrained)

        # Decoder: each block receives (in_ch from below, skip_ch from encoder, out_ch)
        self.decoder4 = DecoderBlock(512, 256, 256)  # /32 -> /16
        self.decoder3 = DecoderBlock(256, 128, 128)  # /16 -> /8
        self.decoder2 = DecoderBlock(128,  64,  64)  # /8  -> /4
        self.decoder1 = DecoderBlock( 64,  64,  64)  # /4  -> /2

        self.head = nn.Conv2d(64, num_classes, kernel_size=1)  # <1>

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape

        # Encoder: extract multi-scale features
        s0, s1, s2, s3, s4 = self.encoder(x)
        # s0: (B, 64,  H/2,  W/2)
        # s1: (B, 64,  H/4,  W/4)
        # s2: (B, 128, H/8,  W/8)
        # s3: (B, 256, H/16, W/16)
        # s4: (B, 512, H/32, W/32)

        # Decoder: upsample + skip concatenation
        d4 = self.decoder4(s4, s3)   # (B, 256, H/16, W/16)  # <2>
        d3 = self.decoder3(d4, s2)   # (B, 128, H/8,  W/8)
        d2 = self.decoder2(d3, s1)   # (B, 64,  H/4,  W/4)
        d1 = self.decoder1(d2, s0)   # (B, 64,  H/2,  W/2)

        # Final upsample to input resolution + classification head
        d0 = F.interpolate(d1, size=(H, W), mode="bilinear", align_corners=False)  # <3>
        return self.head(d0)          # (B, num_classes, H, W)


# Sanity check: verify output shape and print parameter count
model = UNet(num_classes=NUM_CLASSES, pretrained=False)
dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE)
out   = model(dummy)
print(f"Input:  {tuple(dummy.shape)}")
print(f"Output: {tuple(out.shape)}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

1. A $1 \times 1$ convolution maps the 64-channel feature map to `num_classes` logits. No activation — we return raw logits for use with `F.cross_entropy`.
2. The decoder proceeds from the bottleneck (coarsest features, $H/32$) upward. Each block takes the previous decoder output and the corresponding encoder skip, doubling the spatial resolution at each step.
3. After the four decoder blocks we are at $H/2$ resolution. A final bilinear upsample restores the full input resolution — this last step does not carry a skip connection.

## Loss functions

**Cross-entropy** for segmentation is applied pixel-wise. Given logits $\mathbf{z}_{ij} \in \mathbb{R}^C$ at pixel $(i, j)$ and ground-truth class $y_{ij}$:

$$
\mathcal{L}_\text{CE} = -\frac{1}{HW}\sum_{i=1}^H\sum_{j=1}^W \log p_{y_{ij}}(i, j)
$$

where $\mathbf{p}(i,j) = \text{Softmax}(\mathbf{z}_{ij}).$ With strong class imbalance — background pixels typically dominate — the model can achieve low cross-entropy loss by concentrating on the most frequent class, while minority classes remain poorly predicted.

<br>

**Dice loss** addresses this by optimizing an overlap measure directly. The **Dice similarity coefficient** between two binary sets $A$ and $B$ is $\text{DSC}(A, B) = 2|A \cap B| / (|A| + |B|)$. For soft predictions (class probability $p_c$ and one-hot target $y_c$ for class $c$, summed over all pixels):

$$
\text{Dice}_c = \frac{2\sum_{i,j} p_c(i,j)\, y_c(i,j) + \varepsilon}{\sum_{i,j} p_c(i,j) + \sum_{i,j} y_c(i,j) + \varepsilon}
$$

where $\varepsilon > 0$ prevents division by zero when a class is absent. The loss is:

$$
\mathcal{L}_\text{Dice} = 1 - \frac{1}{C}\sum_{c=1}^C \text{Dice}_c
$$

Because each class $c$ contributes equally to $\mathcal{L}_\text{Dice}$ regardless of how many pixels it occupies, the loss gives the same gradient signal to minority and majority classes alike. Combining both objectives, $\mathcal{L} = \mathcal{L}_\text{CE} + \mathcal{L}_\text{Dice}$, gives the model both pixel-wise calibration (from cross-entropy) and class-balanced region overlap (from Dice).

:::{.callout-note}
The $\varepsilon$ term is added to both numerator and denominator — not just the denominator. Adding it to the numerator as well prevents $\text{Dice}_c = 0$ when both prediction and ground truth are zero (the class is absent), which would incorrectly penalize the model.

:::

Implementing Dice loss and the combined segmentation loss:

In [ ]:
def dice_loss(logits: torch.Tensor, targets: torch.Tensor,
              num_classes: int, eps: float = 1.0) -> torch.Tensor:
    """Soft multiclass Dice loss averaged over classes."""
    probs = F.softmax(logits, dim=1)               # <1>

    # One-hot encode targets: (B, H, W) -> (B, C, H, W)
    B, C, H, W = probs.shape
    one_hot = torch.zeros_like(probs)              # <2>
    one_hot.scatter_(1, targets.unsqueeze(1), 1.0)

    # Flatten spatial dimensions for each class
    probs   = probs.view(B, C, -1)                 # (B, C, H*W)
    one_hot = one_hot.view(B, C, -1)               # (B, C, H*W)

    # Numerator: 2 * sum(p * y) + eps; denominator: sum(p) + sum(y) + eps
    num   = 2.0 * (probs * one_hot).sum(dim=2) + eps   # (B, C)  # <3>
    denom = probs.sum(dim=2) + one_hot.sum(dim=2) + eps # (B, C)

    dice_per_class = (num / denom).mean(dim=0)    # average over batch -> (C,)  # <4>
    return 1.0 - dice_per_class.mean()            # scalar loss


def segmentation_loss(logits: torch.Tensor, targets: torch.Tensor,
                      num_classes: int) -> torch.Tensor:
    """Combined cross-entropy + Dice loss."""
    return F.cross_entropy(logits, targets) + dice_loss(logits, targets, num_classes)

1. Apply softmax over the class dimension to convert logits to probabilities $p_c \in [0, 1]$ with $\sum_c p_c = 1.$
2. `scatter_` fills the one-hot tensor: at each spatial location $(i,j)$, it sets `one_hot[b, y[b,i,j], i, j] = 1.0` for each batch element $b.$
3. Dice numerator and denominator are computed per sample and per class by summing over flattened spatial dimension.
4. Average Dice first over the batch dimension, then over classes, to get $\mathcal{L}_\text{Dice} = 1 - \frac{1}{C}\sum_c \text{Dice}_c.$

## Training

**Training.** We train UNet with a pretrained ResNet-18 encoder using AdamW with learning rate $10^{-3}$ and OneCycleLR scheduling over 30 epochs. Using a pretrained encoder allows the model to leverage ImageNet features for the early layers, reducing the number of epochs needed to converge on the relatively small Pet dataset.

In [ ]:
#| output: false
EPOCHS = 30
LR     = 1e-3

torch.manual_seed(RANDOM_SEED)
model = UNet(num_classes=NUM_CLASSES, pretrained=True).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = OneCycleLR(
    optimizer, max_lr=LR,
    steps_per_epoch=len(train_loader), epochs=EPOCHS
)

history = {"train_loss": [], "val_miou": []}

for epoch in range(1, EPOCHS + 1):
    # --- Training ---
    model.train()
    total_loss = 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        logits = model(imgs)
        loss   = segmentation_loss(logits, masks, NUM_CLASSES)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item() * imgs.size(0)

    train_loss = total_loss / len(train_ds)

    # --- Validation ---
    model.eval()
    all_preds, all_targets = [], []
    with torch.inference_mode():
        for imgs, masks in val_loader:
            imgs = imgs.to(DEVICE)
            preds = model(imgs).argmax(dim=1).cpu()
            all_preds.append(preds)
            all_targets.append(masks)

    all_preds   = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)
    _, val_miou, _ = compute_metrics(all_preds, all_targets, NUM_CLASSES)

    history["train_loss"].append(train_loss)
    history["val_miou"].append(val_miou)

    if epoch % 5 == 0 or epoch == 1:
        print(f"[Epoch {epoch:>3d}/{EPOCHS}]  "
              f"train_loss: {train_loss:.4f}  val_mIoU: {val_miou:.4f}")

**Figure.** Training loss and validation mIoU over 30 epochs:

In [ ]:
#| code-fold: true
#| label: fig-training-curves
#| fig-cap: "Training loss (left) and validation mIoU (right) across 30 epochs. OneCycleLR produces a characteristic warm-up followed by a long decay."

epochs_range = range(1, EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.plot(epochs_range, history["train_loss"], color="C0")
ax1.set_xlabel("epoch")
ax1.set_ylabel("train loss (CE + Dice)")
ax1.grid(linestyle="dotted", alpha=0.6)

ax2.plot(epochs_range, history["val_miou"], color="C1")
ax2.set_xlabel("epoch")
ax2.set_ylabel("val mIoU")
ax2.grid(linestyle="dotted", alpha=0.6)

fig.tight_layout()
plt.show();

## Results

**Figure.** Qualitative results on the test set — original image, ground truth mask, and predicted mask:

In [ ]:
#| code-fold: true
#| label: fig-qualitative
#| fig-cap: "Qualitative segmentation results. Each row shows the input image, the ground-truth trimap, and the predicted mask. Mask colors: gray = background, green = foreground (pet), orange = uncertain/boundary."

model.eval()
img_mean_t = torch.tensor(PetsDataset.IMG_MEAN).view(3, 1, 1)
img_std_t  = torch.tensor(PetsDataset.IMG_STD).view(3, 1, 1)

sample_ids = [0, 5, 10, 15]
fig, axes = plt.subplots(len(sample_ids), 3, figsize=(9, 10))
col_titles = ["Image", "Ground Truth", "Prediction"]

for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=9)

for row, idx in enumerate(sample_ids):
    img_t, mask_t = test_ds[idx]
    with torch.inference_mode():
        pred = model(img_t.unsqueeze(0).to(DEVICE)).argmax(dim=1).squeeze(0).cpu()

    img_np = (img_t * img_std_t + img_mean_t).clamp(0, 1).permute(1, 2, 0).numpy()

    axes[row, 0].imshow(img_np)
    axes[row, 0].axis("off")

    axes[row, 1].imshow(mask_t.numpy(), cmap=MASK_CMAP, vmin=0, vmax=2)
    axes[row, 1].axis("off")

    axes[row, 2].imshow(pred.numpy(), cmap=MASK_CMAP, vmin=0, vmax=2)
    axes[row, 2].axis("off")

fig.tight_layout()
plt.show();

Per-class IoU on the full test set:

In [ ]:
model.eval()
all_preds, all_targets = [], []
with torch.inference_mode():
    for imgs, masks in test_loader:
        preds = model(imgs.to(DEVICE)).argmax(dim=1).cpu()
        all_preds.append(preds)
        all_targets.append(masks)

all_preds   = torch.cat(all_preds)
all_targets = torch.cat(all_targets)
accuracy, miou, iou_per_class = compute_metrics(all_preds, all_targets, NUM_CLASSES)

class_names = ["Background", "Foreground", "Boundary"]
print(f"Pixel accuracy: {accuracy:.4f}")
print(f"mIoU:           {miou:.4f}")
print()
for name, iou in zip(class_names, iou_per_class):
    print(f"  IoU [{name:<12s}]: {iou:.4f}")

## Appendix: Transposed convolutions

A regular convolution $\mathbf{y} = W * \mathbf{x}$ downsamples or preserves spatial resolution. A **transposed convolution** (`ConvTranspose2d`) is the adjoint of this operation and produces upsampled output. To see why, consider representing a convolution as a linear map $W_\text{mat}$ acting on a vectorized input $\mathbf{x}_\text{vec}$:

$$
\mathbf{y}_\text{vec} = W_\text{mat}\, \mathbf{x}_\text{vec}
$$

The backward pass computes $\partial \mathcal{L}/\partial \mathbf{x}_\text{vec} = W_\text{mat}^\top\, (\partial \mathcal{L}/\partial \mathbf{y}_\text{vec})$, i.e. multiplication by the transpose of the convolution matrix. `ConvTranspose2d` computes exactly this transposed operation in the forward pass:

$$
\mathbf{z}_\text{vec} = W_\text{mat}^\top\, \mathbf{x}_\text{vec}.
$$

This is not the mathematical inverse of the convolution, but its **adjoint** (or gradient operation). In practice, with `kernel_size=k` and `stride=s`, `ConvTranspose2d` inserts $s - 1$ zeros between input elements before convolving — this is sometimes called a **fractionally strided convolution**. For `kernel_size=2, stride=2`, a $H \times W$ input maps to an exactly $2H \times 2W$ output, making it a clean $2\times$ upsampler.

**Example.** For a $2 \times 2$ input with a $2 \times 2$ kernel and stride $s=2$:

$$
\underbrace{\begin{bmatrix} a & b \\ c & d \end{bmatrix}}_{2 \times 2}
\xrightarrow{\text{ConvTranspose2d}(k=2, s=2)}
\underbrace{\begin{bmatrix} \cdot & \cdot & \cdot & \cdot \\ \cdot & \cdot & \cdot & \cdot \\ \cdot & \cdot & \cdot & \cdot \\ \cdot & \cdot & \cdot & \cdot \end{bmatrix}}_{4 \times 4}
$$

Each input element $a, b, c, d$ produces a $2 \times 2$ patch in the output (by convolving with the kernel), and the patches are placed non-overlappingly, doubling the spatial resolution in both dimensions.

Verifying output dimensions:

In [ ]:
up = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
x  = torch.randn(1, 64, 7, 7)
print(f"Input:  {tuple(x.shape)}")         # (1, 64, 7, 7)
print(f"Output: {tuple(up(x).shape)}")     # (1, 32, 14, 14)

# Also confirm the decoder block doubles spatial dims
block = DecoderBlock(in_ch=512, skip_ch=256, out_ch=256)
x_low  = torch.randn(2, 512,  8,  8)
x_skip = torch.randn(2, 256, 16, 16)
print(f"\nDecoderBlock input:  {tuple(x_low.shape)}")
print(f"DecoderBlock output: {tuple(block(x_low, x_skip).shape)}")

■